# 02b · Fine-tuned attention-MIL (end-to-end)  [GPU]
The **frozen**-backbone MIL underfit (test AUC ~0.66; controls poorly rejected): ImageNet features don't capture the µCT galleries. This notebook fine-tunes the backbone **end-to-end** with the attention head — the last ~20 non-BN layers are trainable, BatchNorm stays frozen, and the loss is **class-balanced** (78 infested vs 27 control) so the model can't collapse to always-infested.

Heavier than 02b-frozen (72 images/bag flow through the CNN each step). Start with MobileNetV2.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()/'nbpkg'))
import numpy as np, pandas as pd, pickle
from config import CFG
import dataset as ds, eval_core as ec, mil
from sklearn.metrics import roc_auc_score
CFG.out_dir.mkdir(parents=True, exist_ok=True)

## Step 0 — fine-tuning settings
Galleries sit near the fruit border, so more resolution helps. If the M1 runs out of memory, drop `img_size` to 128 and/or keep `ft_batch_bags=1`.

In [ ]:
FT_BACKBONE = 'MobileNetV2'
CFG.img_size = 160        # try 224 for max detail; 128 if OOM
CFG.ft_batch_bags = 1     # bags per step; raise to 2-4 if memory allows
FT_EPOCHS, FT_PATIENCE, FT_LR, N_LAST = 40, 8, 1e-4, 20
print('backbone', FT_BACKBONE, '| img_size', CFG.img_size, '| batch_bags', CFG.ft_batch_bags)

In [ ]:
import random, os, tensorflow as tf
os.environ['PYTHONHASHSEED']=str(CFG.seed); random.seed(CFG.seed); np.random.seed(CFG.seed); tf.random.set_seed(CFG.seed)
fruits = ds.build_fruit_index(CFG.data_root, fruit_key=CFG.fruit_key)[0]
by={f.fruit_id:f for f in fruits}
sp=pd.read_csv(CFG.out_dir/'split_single.csv').set_index('fruit_id')['split']
tr_f=[by[i] for i in sp[sp=='train'].index]; va_f=[by[i] for i in sp[sp=='val'].index]; te_f=[by[i] for i in sp[sp=='test'].index]
print(f'fruits train={len(tr_f)} val={len(va_f)} test={len(te_f)}')

## Step 1 — train end-to-end (early stopping on val AUC, class-balanced)

In [ ]:
model = mil.train_finetune_mil(CFG, FT_BACKBONE, tr_f, va_f,
        epochs=FT_EPOCHS, patience=FT_PATIENCE, lr=FT_LR, n_last=N_LAST, verbose=2)

## Step 2 — predict val & test, quick AUC check

In [ ]:
pva,ava = mil.predict_bags_ft(model, va_f, CFG, FT_BACKBONE)
pte,ate = mil.predict_bags_ft(model, te_f, CFG, FT_BACKBONE)
yva=np.array([f.label for f in va_f]); yte=np.array([f.label for f in te_f])
print(f'val AUC  = {roc_auc_score(yva,pva):.3f}')
print(f'test AUC = {roc_auc_score(yte,pte):.3f}   (frozen MobileNet was ~0.67)')
print(f'test prob: infested mean={pte[yte==1].mean():.2f}  control mean={pte[yte==0].mean():.2f}')

## Step 3 — merge into mil_predictions.pkl (so 03b evaluates it)

In [ ]:
key=f'{FT_BACKBONE}-FT'
try: allp=pickle.load(open(CFG.out_dir/'mil_predictions.pkl','rb'))
except FileNotFoundError: allp={}
allp[key]={'val':{'ids':[f.fruit_id for f in va_f],'y':yva,'probs':pva},
           'test':{'ids':[f.fruit_id for f in te_f],'y':yte,'probs':pte,'attn':ate}}
pickle.dump(allp,open(CFG.out_dir/'mil_predictions.pkl','wb'))
try: model.save(CFG.out_dir/f'mil_ft_{FT_BACKBONE}.keras')
except Exception as e: model.save_weights(str(CFG.out_dir/f'mil_ft_{FT_BACKBONE}.weights.h5')); print('(weights only:',type(e).__name__,')')
print('saved. Now run 03b_mil_eval to get the table with CIs for', key)

If test AUC is now clearly above the frozen ~0.67 and controls separate (control mean prob well below infested), we're on track — run **03b**. If it's still stuck, tell me the numbers: next levers are more unfrozen layers, img_size 224, or DenseNet121.